# Stubborn Man (SM) Strategy Validation & Interactive Visualization

This notebook provides a complete walk-through and validation of the Stubborn Man strategy, Refusal Candle Engine, and structural range trading pipeline.

In [ ]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime, timezone, timedelta
from unittest.mock import MagicMock

# Ensure project root is in path
project_root = os.path.dirname(os.path.dirname(os.path.abspath("__file__")))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from Market_Data_Pipeline.structure_engine import MarketStructureEngine
from Market_Data_Pipeline.supply_demand_engine import SupplyDemandEngine
from Market_Data_Pipeline.state_engine import MarketStateEngine
from Trade_Execution.location_engine import TradeLocationEngine
from Strategies.refusal_candle_engine import RefusalCandleEngine
from Strategies.sm_strategy import SMStrategy
from Visualization.chart_annotator import ChartAnnotationEngine
from Visualization.debug_config import DebugConfig
from ML.ml_decision_engine import MLDecisionEngine
from ML.feature_pipeline import FeaturePipeline

### Phase A: Generate Synthetic Historical Candlesticks with RANGE properties

In [ ]:
# Create deterministic range-bound price action
np.random.seed(42)
n_bars = 400
base_price = 1.1000

# Formulate wave-like range bounds
t = np.linspace(0, 8 * np.pi, n_bars)
trend_wave = 0.0050 * np.sin(t)
noise = np.random.normal(0, 0.0003, n_bars)
closes = base_price + trend_wave + noise

opens = closes.copy()
opens[1:] = closes[:-1]
opens[0] = base_price

highs = np.maximum(opens, closes) + 0.0008
lows = np.minimum(opens, closes) - 0.0008

# Construct candle patterns resembling pin bar / rejections at peaks and troughs
for i in range(1, n_bars):
    # Trough: create long lower wicks (bullish rejection)
    if t[i] % (2*np.pi) > (1.5*np.pi - 0.2) and t[i] % (2*np.pi) < (1.5*np.pi + 0.2):
        lows[i] = min(opens[i], closes[i]) - 0.0025
    # Peak: create long upper wicks (bearish rejection)
    if t[i] % (2*np.pi) > (0.5*np.pi - 0.2) and t[i] % (2*np.pi) < (0.5*np.pi + 0.2):
        highs[i] = max(opens[i], closes[i]) + 0.0025

dates = pd.date_range(end=datetime.now(timezone.utc), periods=n_bars, freq="5min")
df = pd.DataFrame({
    "Datetime": dates,
    "Open": opens,
    "High": highs,
    "Low": lows,
    "Close": closes,
    "TickVolume": np.random.randint(80, 250, n_bars),
    "Spread": np.ones(n_bars)
})
print(f"Generated {len(df)} range-bound bars successfully.")

### Phase B: Run MarketStructureEngine and SupplyDemandEngine

In [ ]:
mse = MarketStructureEngine(lookback=5)
sde = SupplyDemandEngine(atr_period=14, impulse_threshold=1.5)

df_mse = mse.process(df)
df_sde = sde.process(df_mse)
print(f"Detected Swings: {len(mse.swings)}")
print(f"Detected Zones: {len(sde.zones)}")

### Phase C: Validate MarketStateClassifier

In [ ]:
state_eng = MarketStateEngine()
# Create structural graph
last_idx = len(df_sde) - 1
graph = MarketStructureGraph(
    symbol="EURUSD",
    timeframe="M5",
    timestamp=df_sde.iloc[last_idx]["Datetime"],
    swing_highs=[s for s in mse.swings if s.level_type == 'SwingHigh'],
    swing_lows=[s for s in mse.swings if s.level_type == 'SwingLow'],
    protected_high=mse.protected_high,
    protected_low=mse.protected_low,
    bos=list(mse.bos_list),
    choch=list(mse.choch_list),
    supply_zones=[z for z in sde.zones if z.type == 'Supply'],
    demand_zones=[z for z in sde.zones if z.type == 'Demand'],
    trend_direction="Neutral",
    atr=float(df_sde.iloc[last_idx].get("atr_14", 0.001))
)

state_ctx = state_eng.evaluate(graph)
print(f"System predicted market state regime: {state_ctx.regime} with confidence {state_ctx.confidence_score:.2f}")

### Phase D & E: Run RefusalCandleEngine and LevelBreakProbabilityModel

In [ ]:
refusal_eng = RefusalCandleEngine()
print("Running rejection scans...")

rejection_events = []
for idx in range(50, n_bars):
    high_p = df_sde.iloc[idx]["High"]
    low_p = df_sde.iloc[idx]["Low"]
    
    for zone in sde.zones:
        if zone.broken:
            continue
        if zone.type == "Supply" and high_p >= zone.lower:
            res = refusal_eng.evaluate_rejection(df_sde, idx, zone, graph)
            if res.score > 60:
                rejection_events.append((idx, zone, res, "SELL"))
        elif zone.type == "Demand" and low_p <= zone.upper:
            res = refusal_eng.evaluate_rejection(df_sde, idx, zone, graph)
            if res.score > 60:
                rejection_events.append((idx, zone, res, "BUY"))

print(f"Captured {len(rejection_events)} significant rejection events.")

### Phase F: Instantiate and Run SMStrategy Over the Entire Pipeline

In [ ]:
# Instantiate Strategy with mocks to simulate live/backtest runtime pipeline execution
data_feed = MagicMock()
send_order = MagicMock()
trading_journal = MagicMock()
drawdown_manager = MagicMock()
drawdown_manager.trading_allowed.return_value = True

strategy = SMStrategy(
    data_feed=data_feed,
    send_order=send_order,
    trading_journal=trading_journal,
    drawdown_manager=drawdown_manager,
    symbols=["EURUSD"],
    min_refusal_score=60.0,
    max_break_probability=0.40,
    shadow_mode=False
)

# Overwrite decision engine for deterministic outputs in the notebook
decision_engine = MagicMock()
from ML.decision_context import DecisionContext, PolicyRecommendation
mock_ctx = DecisionContext(
    symbol="EURUSD", timeframe="M5", timestamp="2024-01-01",
    predicted_state="RANGE", state_probabilities={"TREND": 0.1, "RANGE": 0.8, "TRANSITION": 0.1},
    state_confidence=0.8, break_probability=0.15, rejection_probability=0.85,
    trade_quality_score=0.8, confidence_score=0.8,
    policy_recommendation=PolicyRecommendation(
        allow_trade=True, suggested_risk_multiplier=1.0, suggested_position_scale=1.0,
        suggested_tp_mode="STRUCTURE_TARGET", suggested_sl_adjustment=0.0
    ),
    model_versions={"LevelBreakProbabilityModel": "v1"}, inference_time_ms=1.0, missing_features=[], warnings=[]
)
decision_engine.evaluate.return_value = mock_ctx
strategy.decision_engine = decision_engine

print("Replaying SMStrategy over synthetic price action to log all decisions and trade levels...")

# Walk forward bar-by-bar starting after the first 50 bars
for idx in range(50, n_bars):
    slice_df = df_sde.iloc[:idx+1]
    strategy._evaluate_setup_and_trade("EURUSD", "M5", slice_df)

print(f"SMStrategy executed with {send_order.execute.call_count} order execution triggers.")

### Phase G: Display Complete Decision Chain and Chart Overlays

In [ ]:
# Set up plot
fig, ax = plt.subplots(figsize=(15, 8))
ax.plot(df_sde.index, df_sde["Close"], color='gray', label='Price', alpha=0.6)

annotator = ChartAnnotationEngine()
annotator.annotate_matplotlib(
    ax=ax,
    msg=graph,
    state_ctx=state_ctx
)

# Plot the strategy execution coordinates from send_order history
for call in send_order.execute.call_args_list:
    args = call[1]
    direction = args["direction"]
    sl = args["sl_price"]
    tp = args["tp_price"]
    
    color = 'green' if direction == 1 else 'red'
    marker = '^' if direction == 1 else 'v'
    
    # Find a reasonable index near the end for plotting
    idx = int(n_bars * 0.75)
    ax.axhline(y=sl, color=color, linestyle='--', alpha=0.5, label=f"SL: {sl:.5f}")
    ax.axhline(y=tp, color=color, linestyle=':', alpha=0.5, label=f"TP: {tp:.5f}")

plt.title("SM Strategy Rejection Setup Verification & Range Analysis", fontsize=14)
plt.xlabel("Bar Index")
plt.ylabel("Price")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()